# 07 Train WLASL1000 Small Transformer Encoder

## Purpose
This notebook tests whether a small Transformer Encoder can improve WLASL1000.

## Why try Transformer?
Transformers use self-attention to compare all frames in a sequence. This can help recognise signs where the overall motion pattern matters.

## Important expectation
This is an experiment. The earlier WLASL100 Transformer was weaker than BiGRU, so this notebook uses a small Transformer to reduce overfitting and GPU memory issues.

In [1]:
from pathlib import Path
import json, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore", category=UserWarning)

e:\Be_My_Ear\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Setup paths and Transformer configuration

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path("E:/Be_My_Ear")
DATASET_NAME = "WLASL1000"
PREFIX = "wlasl1000"

BASE_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
CLEAN_INDEX_FILE = BASE_DIR / f"{PREFIX}_clean_keypoint_index.csv"
LABEL_MAP_FILE = PROJECT_ROOT / "data" / "label_maps" / DATASET_NAME / f"asl_{PREFIX}_labels.json"

MODEL_DIR = PROJECT_ROOT / "models" / "ASL" / DATASET_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)

REPORT_DIR = PROJECT_ROOT / "reports" / f"phase1_{PREFIX}"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Clean index:", CLEAN_INDEX_FILE.exists())

MODEL_NAME = "transformer_encoder"
MODEL_DISPLAY_NAME = "Small Transformer Encoder"
TRAINING_TITLE = "Be My Ear - WLASL1000 Small Transformer Encoder"

MODEL_PATH = MODEL_DIR / f"{MODEL_NAME}_{PREFIX}.pt"
HISTORY_PATH = MODEL_DIR / f"{MODEL_NAME}_{PREFIX}_history.csv"
NORM_STATS_PATH = MODEL_DIR / f"{PREFIX}_transformer_train_norm_stats.npz"
RESULT_FILE = MODEL_DIR / f"{MODEL_NAME}_{PREFIX}_result_summary.csv"

BATCH_SIZE = 16
EPOCHS = 70
EARLY_STOPPING_PATIENCE = 14

USE_VELOCITY = True
INPUT_SIZE = 516
SEQUENCE_LENGTH = 60

D_MODEL = 192
NHEAD = 6
NUM_ENCODER_LAYERS = 3
DIM_FEEDFORWARD = 384
DROPOUT = 0.25

print("Model path:", MODEL_PATH)

Device: cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Clean index: True
Model path: E:\Be_My_Ear\models\ASL\WLASL1000\transformer_encoder_wlasl1000.pt


## 2. Load clean dataset and split data

In [3]:
df = pd.read_csv(CLEAN_INDEX_FILE)

train_records, val_records, test_records = [], [], []

for label_id, group in df.groupby("label_id"):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n = len(group)
    n_test = max(1, int(round(n * 0.15)))
    n_val = max(1, int(round(n * 0.15)))

    test_records.append(group.iloc[:n_test])
    val_records.append(group.iloc[n_test:n_test + n_val])
    train_records.append(group.iloc[n_test + n_val:])

train_df = pd.concat(train_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(val_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat(test_records).sample(frac=1, random_state=SEED).reset_index(drop=True)

NUM_CLASSES = df["label_id"].nunique()

print("Clean samples:", len(df))
print("Classes:", NUM_CLASSES)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

Clean samples: 7232
Classes: 1000
Train: 5024 Val: 1104 Test: 1104


## 3. Compute normalisation using train set only

In [4]:
def compute_train_normalisation_stats(train_dataframe):
    total_sum = None
    total_sq_sum = None
    total_count = 0

    for path in tqdm(train_dataframe["keypoint_path"], desc="Computing train mean/std"):
        arr = np.load(path).astype(np.float32)

        if total_sum is None:
            total_sum = arr.sum(axis=0)
            total_sq_sum = (arr ** 2).sum(axis=0)
        else:
            total_sum += arr.sum(axis=0)
            total_sq_sum += (arr ** 2).sum(axis=0)

        total_count += arr.shape[0]

    mean = total_sum / total_count
    variance = (total_sq_sum / total_count) - (mean ** 2)
    variance = np.maximum(variance, 1e-6)
    std = np.sqrt(variance)
    return mean.astype(np.float32), std.astype(np.float32)

train_mean, train_std = compute_train_normalisation_stats(train_df)
np.savez(NORM_STATS_PATH, mean=train_mean, std=train_std)

print("Saved norm stats:", NORM_STATS_PATH)

Computing train mean/std: 100%|██████████| 5024/5024 [00:01<00:00, 4377.28it/s]

Saved norm stats: E:\Be_My_Ear\models\ASL\WLASL1000\wlasl1000_transformer_train_norm_stats.npz


## 4. Dataset for Transformer

In [5]:
class WLASLTransformerDataset(Dataset):
    def __init__(self, dataframe, mean, std):
        self.dataframe = dataframe.reset_index(drop=True)
        self.mean = mean.reshape(1, -1).astype(np.float32)
        self.std = std.reshape(1, -1).astype(np.float32)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        keypoints = np.load(row["keypoint_path"]).astype(np.float32)
        keypoints = (keypoints - self.mean) / (self.std + 1e-6)

        velocity = np.zeros_like(keypoints, dtype=np.float32)
        velocity[1:] = keypoints[1:] - keypoints[:-1]
        features = np.concatenate([keypoints, velocity], axis=1).astype(np.float32)

        label = int(row["label_id"])
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

## 5. Create balanced data loaders

In [6]:
train_dataset = WLASLTransformerDataset(train_df, train_mean, train_std)
val_dataset = WLASLTransformerDataset(val_df, train_mean, train_std)
test_dataset = WLASLTransformerDataset(test_df, train_mean, train_std)

class_counts = train_df["label_id"].value_counts().to_dict()
sample_weights = train_df["label_id"].map(lambda label: 1.0 / class_counts[label]).values
sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

x_batch, y_batch = next(iter(train_loader))
print("Input batch shape:", x_batch.shape)

Input batch shape: torch.Size([16, 60, 516])


## 6. Positional encoding

In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=60, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 0:
            pe[:, 1::2] = torch.cos(position * div_term)
        else:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

## 7. Define Small Transformer Encoder

In [8]:
class SmallTransformerEncoder(nn.Module):
    def __init__(self, input_size, d_model, num_classes, nhead=6, num_layers=3, dim_feedforward=384, dropout=0.25, sequence_length=60):
        super().__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.positional_encoding = PositionalEncoding(d_model, max_len=sequence_length, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        x = self.input_projection(x)
        x = self.positional_encoding(x)
        encoded = self.encoder(x)

        mean_pool = encoded.mean(dim=1)
        max_pool, _ = encoded.max(dim=1)
        combined = torch.cat([mean_pool, max_pool], dim=1)

        return self.classifier(combined)

def build_model_from_checkpoint(checkpoint):
    return SmallTransformerEncoder(
        input_size=checkpoint.get("input_size", INPUT_SIZE),
        d_model=checkpoint.get("d_model", D_MODEL),
        num_classes=checkpoint.get("num_classes", NUM_CLASSES),
        nhead=checkpoint.get("nhead", NHEAD),
        num_layers=checkpoint.get("num_encoder_layers", NUM_ENCODER_LAYERS),
        dim_feedforward=checkpoint.get("dim_feedforward", DIM_FEEDFORWARD),
        dropout=checkpoint.get("dropout", DROPOUT),
        sequence_length=checkpoint.get("sequence_length", SEQUENCE_LENGTH)
    )

## 8. Initialise Transformer model

In [9]:
model = SmallTransformerEncoder(
    input_size=INPUT_SIZE,
    d_model=D_MODEL,
    num_classes=NUM_CLASSES,
    nhead=NHEAD,
    num_layers=NUM_ENCODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    sequence_length=SEQUENCE_LENGTH
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

print("Parameters:", sum(p.numel() for p in model.parameters()))

def create_checkpoint_payload(epoch, best_val_f1, best_val_top5):
    return {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_f1": best_val_f1,
        "best_val_top5": best_val_top5,
        "num_classes": NUM_CLASSES,
        "input_size": INPUT_SIZE,
        "sequence_length": SEQUENCE_LENGTH,
        "use_velocity": USE_VELOCITY,
        "architecture": "SmallTransformerEncoder",
        "d_model": D_MODEL,
        "nhead": NHEAD,
        "num_encoder_layers": NUM_ENCODER_LAYERS,
        "dim_feedforward": DIM_FEEDFORWARD,
        "dropout": DROPOUT
    }

Parameters: 1258024


## 9. Training helper functions

In [10]:
def top_k_accuracy(outputs, labels, k=5):
    _, top_k_preds = outputs.topk(k, dim=1)
    return top_k_preds.eq(labels.view(-1, 1).expand_as(top_k_preds)).any(dim=1).float().mean().item()

def run_epoch(model, loader, criterion, optimizer=None, phase="Train", epoch=1, total_epochs=1):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = total_top1 = total_top3 = total_top5 = 0
    all_preds, all_labels = [], []

    bar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [{phase}]", leave=False)

    with torch.set_grad_enabled(is_train):
        for step, (x, y) in enumerate(bar, start=1):
            x, y = x.to(device), y.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(x)
            loss = criterion(outputs, y)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            preds = torch.argmax(outputs, dim=1)

            total_loss += loss.item()
            total_top1 += (preds == y).float().mean().item()
            total_top3 += top_k_accuracy(outputs, y, 3)
            total_top5 += top_k_accuracy(outputs, y, 5)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(y.detach().cpu().numpy())

            bar.set_postfix({
                "step": f"{step}/{len(loader)}",
                "loss": f"{loss.item():.4f}",
                "top1": f"{(preds == y).float().mean().item():.4f}",
                "top5": f"{top_k_accuracy(outputs, y, 5):.4f}"
            })

    return (
        total_loss / len(loader),
        total_top1 / len(loader),
        total_top3 / len(loader),
        total_top5 / len(loader),
        f1_score(all_labels, all_preds, average="macro", zero_division=0)
    )

def top_k_accuracy_numpy(y_true, y_probs, k):
    correct = 0
    for true_label, prob in zip(y_true, y_probs):
        if true_label in np.argsort(prob)[-k:]:
            correct += 1
    return correct / len(y_true)

def collect_predictions(model, loader):
    model.eval()
    labels, preds, probs_all = [], [], []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Collecting predictions"):
            x = x.to(device)
            outputs = model(x)
            probs = F.softmax(outputs, dim=1)
            pred = torch.argmax(probs, dim=1)

            labels.extend(y.numpy())
            preds.extend(pred.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())

    return np.array(labels), np.array(preds), np.array(probs_all)

## 10. Train Transformer model

In [11]:
history = {k: [] for k in [
    "train_loss", "train_top1", "train_top3", "train_top5", "train_f1",
    "val_loss", "val_top1", "val_top3", "val_top5", "val_f1", "lr"
]}

best_val_f1 = 0.0
best_val_top5 = 0.0
epochs_without_improvement = 0

print("=" * 80)
print(TRAINING_TITLE)
print("=" * 80)
print("Device:", device)
print("Input shape:", (60, INPUT_SIZE))
print("Classes:", NUM_CLASSES)
print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Model path:", MODEL_PATH)
print("=" * 80)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 80)

    train_loss, train_top1, train_top3, train_top5, train_f1 = run_epoch(
        model, train_loader, criterion, optimizer=optimizer, phase="Training", epoch=epoch, total_epochs=EPOCHS
    )

    val_loss, val_top1, val_top3, val_top5, val_f1 = run_epoch(
        model, val_loader, criterion, optimizer=None, phase="Validation", epoch=epoch, total_epochs=EPOCHS
    )

    scheduler.step(val_f1)
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["train_top1"].append(train_top1)
    history["train_top3"].append(train_top3)
    history["train_top5"].append(train_top5)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_top1"].append(val_top1)
    history["val_top3"].append(val_top3)
    history["val_top5"].append(val_top5)
    history["val_f1"].append(val_f1)
    history["lr"].append(current_lr)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_top5 = val_top5
        epochs_without_improvement = 0

        payload = create_checkpoint_payload(epoch, best_val_f1, best_val_top5)
        torch.save(payload, MODEL_PATH)
        save_status = "Saved new best model"
    else:
        epochs_without_improvement += 1
        save_status = "No improvement"

    print(f"Train | Loss: {train_loss:.4f} | Top-1: {train_top1:.4f} | Top-3: {train_top3:.4f} | Top-5: {train_top5:.4f} | F1: {train_f1:.4f}")
    print(f"Val   | Loss: {val_loss:.4f} | Top-1: {val_top1:.4f} | Top-3: {val_top3:.4f} | Top-5: {val_top5:.4f} | F1: {val_f1:.4f}")
    print(f"Learning rate: {current_lr:.8f}")
    print("Status:", save_status)
    print(f"Best Val F1 so far: {best_val_f1:.4f}")
    print(f"Best Val Top-5 so far: {best_val_top5:.4f}")
    print(f"Epochs without improvement: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nTraining completed in minutes:", round((time.time() - start_time) / 60, 2))
print("Best model saved:", MODEL_PATH)

Be My Ear - WLASL1000 Small Transformer Encoder
Device: cuda
Input shape: (60, 516)
Classes: 1000
Train samples: 5024
Validation samples: 1104
Test samples: 1104
Epochs: 70
Batch size: 16
Model path: E:\Be_My_Ear\models\ASL\WLASL1000\transformer_encoder_wlasl1000.pt

Epoch 1/70
--------------------------------------------------------------------------------


Train | Loss: 6.9074 | Top-1: 0.0014 | Top-3: 0.0062 | Top-5: 0.0090 | F1: 0.0002
Val   | Loss: 6.8065 | Top-1: 0.0027 | Top-3: 0.0082 | Top-5: 0.0154 | F1: 0.0000
Learning rate: 0.00020000
Status: Saved new best model
Best Val F1 so far: 0.0000
Best Val Top-5 so far: 0.0154
Epochs without improvement: 0/14

Epoch 2/70
--------------------------------------------------------------------------------


Train | Loss: 6.6671 | Top-1: 0.0048 | Top-3: 0.0119 | Top-5: 0.0161 | F1: 0.0009
Val   | Loss: 6.6176 | Top-1: 0.0054 | Top-3: 0.0127 | Top-5: 0.0217 | F1: 0.0004
Learning rate: 0.00020000
Status: Saved new best model
Best Val F1 so far: 0.0004
Best Val Top-5 so far: 0.0217
Epochs without improvement: 0/14

Epoch 3/70
--------------------------------------------------------------------------------


Train | Loss: 6.4374 | Top-1: 0.0088 | Top-3: 0.0235 | Top-5: 0.0370 | F1: 0.0021
Val   | Loss: 6.4274 | Top-1: 0.0100 | Top-3: 0.0236 | Top-5: 0.0408 | F1: 0.0025
Learning rate: 0.00020000
Status: Saved new best model
Best Val F1 so far: 0.0025
Best Val Top-5 so far: 0.0408
Epochs without improvement: 0/14

Epoch 4/70
--------------------------------------------------------------------------------


Train | Loss: 6.1729 | Top-1: 0.0223 | Top-3: 0.0502 | Top-5: 0.0744 | F1: 0.0075
Val   | Loss: 6.2342 | Top-1: 0.0136 | Top-3: 0.0326 | Top-5: 0.0471 | F1: 0.0064
Learning rate: 0.00020000
Status: Saved new best model
Best Val F1 so far: 0.0064
Best Val Top-5 so far: 0.0471
Epochs without improvement: 0/14

Epoch 5/70
--------------------------------------------------------------------------------


...

Train | Loss: 0.9542 | Top-1: 0.9596 | Top-3: 0.9986 | Top-5: 0.9996 | F1: 0.9524
Val   | Loss: 4.3069 | Top-1: 0.2355 | Top-3: 0.4031 | Top-5: 0.4918 | F1: 0.1790
Learning rate: 0.00005000
Status: No improvement
Best Val F1 so far: 0.1887
Best Val Top-5 so far: 0.5000
Epochs without improvement: 7/14

Epoch 67/70
--------------------------------------------------------------------------------


Train | Loss: 0.9470 | Top-1: 0.9612 | Top-3: 0.9986 | Top-5: 0.9998 | F1: 0.9566
Val   | Loss: 4.2904 | Top-1: 0.2319 | Top-3: 0.4130 | Top-5: 0.5036 | F1: 0.1746
Learning rate: 0.00005000
Status: No improvement
Best Val F1 so far: 0.1887
Best Val Top-5 so far: 0.5000
Epochs without improvement: 8/14

Epoch 68/70
--------------------------------------------------------------------------------


Train | Loss: 0.9379 | Top-1: 0.9664 | Top-3: 0.9994 | Top-5: 0.9998 | F1: 0.9606
Val   | Loss: 4.2549 | Top-1: 0.2409 | Top-3: 0.4248 | Top-5: 0.5000 | F1: 0.1835
Learning rate: 0.00005000
Status: No improvement
Best Val F1 so far: 0.1887
Best Val Top-5 so far: 0.5000
Epochs without improvement: 9/14

Epoch 69/70
--------------------------------------------------------------------------------


Train | Loss: 0.9363 | Top-1: 0.9626 | Top-3: 0.9982 | Top-5: 0.9998 | F1: 0.9617
Val   | Loss: 4.2823 | Top-1: 0.2328 | Top-3: 0.4149 | Top-5: 0.4909 | F1: 0.1778
Learning rate: 0.00005000
Status: No improvement
Best Val F1 so far: 0.1887
Best Val Top-5 so far: 0.5000
Epochs without improvement: 10/14

Epoch 70/70
--------------------------------------------------------------------------------


Train | Loss: 0.9229 | Top-1: 0.9660 | Top-3: 0.9994 | Top-5: 1.0000 | F1: 0.9602
Val   | Loss: 4.2806 | Top-1: 0.2391 | Top-3: 0.4130 | Top-5: 0.4982 | F1: 0.1826
Learning rate: 0.00005000
Status: No improvement
Best Val F1 so far: 0.1887
Best Val Top-5 so far: 0.5000
Epochs without improvement: 11/14

Training completed in minutes: 10.13
Best model saved: E:\Be_My_Ear\models\ASL\WLASL1000\transformer_encoder_wlasl1000.pt


## 11. Save history and evaluate Transformer

In [12]:
history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)
print("Saved history:", HISTORY_PATH)

checkpoint = torch.load(MODEL_PATH, map_location=device)
model = build_model_from_checkpoint(checkpoint).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred, y_probs = collect_predictions(model, test_loader)

test_top1 = accuracy_score(y_true, y_pred)
test_top3 = top_k_accuracy_numpy(y_true, y_probs, 3)
test_top5 = top_k_accuracy_numpy(y_true, y_probs, 5)
test_macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("=" * 80)
print(f"{MODEL_DISPLAY_NAME} Test Evaluation")
print("=" * 80)
print(f"Test Top-1 Accuracy: {test_top1:.4f}")
print(f"Test Top-3 Accuracy: {test_top3:.4f}")
print(f"Test Top-5 Accuracy: {test_top5:.4f}")
print(f"Test Macro F1: {test_macro_f1:.4f}")

result_df = pd.DataFrame([{
    "dataset": DATASET_NAME,
    "model": MODEL_DISPLAY_NAME,
    "clean_samples": len(df),
    "classes": NUM_CLASSES,
    "input_shape": f"(60, {INPUT_SIZE})",
    "features": "keypoints + velocity",
    "best_val_f1": checkpoint["best_val_f1"],
    "best_val_top5": checkpoint["best_val_top5"],
    "checkpoint_epoch": checkpoint["epoch"],
    "test_top1_accuracy": test_top1,
    "test_top3_accuracy": test_top3,
    "test_top5_accuracy": test_top5,
    "test_macro_f1": test_macro_f1,
    "model_path": str(MODEL_PATH),
    "history_path": str(HISTORY_PATH),
    "norm_stats_path": str(NORM_STATS_PATH)
}])

result_df.to_csv(RESULT_FILE, index=False)
report_result_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_result_summary.csv"
result_df.to_csv(report_result_file, index=False)

print("Saved result summary:", RESULT_FILE)
display(result_df)

Saved history: E:\Be_My_Ear\models\ASL\WLASL1000\transformer_encoder_wlasl1000_history.csv


Small Transformer Encoder Test Evaluation
Test Top-1 Accuracy: 0.2210
Test Top-3 Accuracy: 0.4004
Test Top-5 Accuracy: 0.4891
Test Macro F1: 0.1666
Saved result summary: E:\Be_My_Ear\models\ASL\WLASL1000\transformer_encoder_wlasl1000_result_summary.csv


,dataset,model,clean_samples,classes,input_shape,features,best_val_f1,best_val_top5,checkpoint_epoch,test_top1_accuracy,test_top3_accuracy,test_top5_accuracy,test_macro_f1,model_path,history_path,norm_stats_path
0,WLASL1000,Small Transformer Encoder,7232,1000,"(60, 516)",keypoints + velocity,0.188702,0.5,59,0.221014,0.400362,0.48913,0.166631,E:\Be_My_Ear\models\ASL\WLASL1000\transformer_...,E:\Be_My_Ear\models\ASL\WLASL1000\transformer_...,E:\Be_My_Ear\models\ASL\WLASL1000\wlasl1000_tr...


## 12. Confidence threshold analysis

In [13]:
if LABEL_MAP_FILE.exists():
    with open(LABEL_MAP_FILE, "r", encoding="utf-8") as f:
        label_map = json.load(f)
    id_to_gloss = {int(k): v["gloss"] for k, v in label_map.items()}
else:
    id_to_gloss = {}

test_df_reset = test_df.reset_index(drop=True)
prediction_records = []

for i in range(len(y_true)):
    true_id = int(y_true[i])
    pred_id = int(y_pred[i])
    confidence = float(y_probs[i][pred_id])
    top5_ids = np.argsort(y_probs[i])[-5:][::-1]

    prediction_records.append({
        "video_id": test_df_reset.iloc[i]["video_id"],
        "true_label_id": true_id,
        "true_gloss": id_to_gloss.get(true_id, str(true_id)),
        "predicted_label_id": pred_id,
        "predicted_gloss": id_to_gloss.get(pred_id, str(pred_id)),
        "confidence": confidence,
        "correct_top1": true_id == pred_id,
        "correct_top5": true_id in top5_ids,
        "top5_glosses": ", ".join([id_to_gloss.get(int(x), str(x)) for x in top5_ids])
    })

predictions_df = pd.DataFrame(prediction_records)
predictions_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_test_predictions.csv"
predictions_df.to_csv(predictions_file, index=False)

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
threshold_records = []

for threshold in thresholds:
    confident = predictions_df[predictions_df["confidence"] >= threshold]
    threshold_records.append({
        "confidence_threshold": threshold,
        "coverage": len(confident) / len(predictions_df),
        "top1_accuracy_on_confident_samples": confident["correct_top1"].mean() if len(confident) else np.nan,
        "top5_accuracy_on_confident_samples": confident["correct_top5"].mean() if len(confident) else np.nan,
        "num_confident_samples": len(confident)
    })

threshold_df = pd.DataFrame(threshold_records)
threshold_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_confidence_threshold_analysis.csv"
threshold_df.to_csv(threshold_file, index=False)

print("Saved predictions:", predictions_file)
print("Saved confidence threshold analysis:", threshold_file)
display(threshold_df)

Saved predictions: E:\Be_My_Ear\reports\phase1_wlasl1000\transformer_encoder_wlasl1000_test_predictions.csv
Saved confidence threshold analysis: E:\Be_My_Ear\reports\phase1_wlasl1000\transformer_encoder_wlasl1000_confidence_threshold_analysis.csv


,confidence_threshold,coverage,top1_accuracy_on_confident_samples,top5_accuracy_on_confident_samples,num_confident_samples
0,0.2,0.768116,0.255896,0.547170,848
1,0.3,0.533514,0.307301,0.594228,589
2,0.4,0.375000,0.350242,0.637681,414
3,0.5,0.255435,0.386525,0.670213,282
4,0.6,0.180254,0.447236,0.713568,199
5,0.7,0.116848,0.457364,0.720930,129
6,0.8,0.062500,0.449275,0.666667,69
7,0.9,0.022645,0.440000,0.680000,25


## 13. Compare V1 vs Transformer

In [14]:
v1_file = MODEL_DIR / f"bigru_attention_{PREFIX}_result_summary.csv"
rows = []

if v1_file.exists():
    v1 = pd.read_csv(v1_file).iloc[0].to_dict()
    rows.append({
        "version": "V1",
        "model": v1.get("model", "BiGRU + Temporal Attention"),
        "test_top1_accuracy": float(v1.get("test_top1_accuracy", np.nan)),
        "test_top3_accuracy": float(v1.get("test_top3_accuracy", np.nan)),
        "test_top5_accuracy": float(v1.get("test_top5_accuracy", np.nan)),
        "test_macro_f1": float(v1.get("test_macro_f1", np.nan)),
        "best_val_f1": float(v1.get("best_val_f1", np.nan)),
        "best_val_top5": float(v1.get("best_val_top5", np.nan)),
        "source": str(v1_file)
    })

rows.append({
    "version": "Transformer",
    "model": MODEL_DISPLAY_NAME,
    "test_top1_accuracy": test_top1,
    "test_top3_accuracy": test_top3,
    "test_top5_accuracy": test_top5,
    "test_macro_f1": test_macro_f1,
    "best_val_f1": checkpoint["best_val_f1"],
    "best_val_top5": checkpoint["best_val_top5"],
    "source": str(RESULT_FILE)
})

comparison_df = pd.DataFrame(rows)
comparison_file = REPORT_DIR / f"{PREFIX}_v1_vs_transformer_comparison.csv"
comparison_df.to_csv(comparison_file, index=False)

print("Saved comparison:", comparison_file)
display(comparison_df)

Saved comparison: E:\Be_My_Ear\reports\phase1_wlasl1000\wlasl1000_v1_vs_transformer_comparison.csv


,version,model,test_top1_accuracy,test_top3_accuracy,test_top5_accuracy,test_macro_f1,best_val_f1,best_val_top5,source
0,V1,BiGRU + Temporal Attention,0.208036,0.388393,0.463393,0.167371,0.170589,0.463393,E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attent...
1,Transformer,Small Transformer Encoder,0.221014,0.400362,0.489130,0.166631,0.188702,0.500000,E:\Be_My_Ear\models\ASL\WLASL1000\transformer_...
